# TASK D — step-50 GRPO checkpoint on the TEST split (one vLLM session)

Evaluates **base / grpo-ckpt50 / grpo-final** on the 162-puzzle held-out test
split, greedy, all three arms through ONE vLLM session so every comparison is
within-session. Then runs the paired bootstrap (repo estimator, seed 0).

**Settings → Accelerator → GPU T4 x2, Internet → On.** `HF_TOKEN` must exist as
a Kaggle secret (read access to the private `-ckpt` repo). Total ~1 h.

Prerequisite: the `analysis/aug20` branch is pushed to GitHub (cell 1 checks
out that branch; it fails loudly if the push hasn't happened).

Serving flags are copied verbatim from `notebooks/kaggle_eval_7b.ipynb` — the
session-A recipe that already worked on this hardware — with only the LoRA
modules swapped. Outputs land under `results-analysis/aug20/taskD-session/`;
nothing existing is touched. When cell 3 finishes, grab
`/kaggle/working/taskD-session.zip` (cell 4 also pushes a durable copy to the
Hub results dataset).

In [ ]:
# Cell 1 — setup: repo @ analysis/aug20, data, the two adapters
import os, sys
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!git checkout analysis/aug20
assert os.path.exists('results-analysis/aug20/taskD_eval_ckpt50.yaml'), \
    'analysis/aug20 branch missing or not pushed — push it from the Mac first'
!pip install -q -e . openai vllm
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b-ckpt',
                  local_dir='adapters/grpo-7b-ckpt', token=os.environ['HF_TOKEN'])
snapshot_download(f'{HF_USER}/connections-rl-grpo-7b',
                  local_dir='adapters/grpo-7b', token=os.environ['HF_TOKEN'])
!ls adapters/grpo-7b-ckpt
CKPT50 = 'adapters/grpo-7b-ckpt/checkpoint-50'
assert os.path.isdir(CKPT50) and os.path.exists(f'{CKPT50}/adapter_config.json'), (
    'checkpoint-50/ not found (or not a PEFT adapter dir) in the ckpt repo — '
    'STOP HERE and report the actual layout; do not substitute another checkpoint.')
print('checkpoint-50 OK')

In [ ]:
# Cell 2 — vLLM: session-A serving recipe, only the LoRA modules swapped
import subprocess, time, urllib.request

A = '/kaggle/working/connections-rl/adapters'
# --enforce-eager: CUDA-graph capture OOMs on 14.5GB T4s with 7B+LoRA (see
# notebooks/kaggle_eval_7b.ipynb, which this command mirrors verbatim).
proc = subprocess.Popen(
    f'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
    f'--enable-lora --enforce-eager '
    f'--lora-modules connections-rl-grpo-7b-ckpt50={A}/grpo-7b-ckpt/checkpoint-50 '
    f'connections-rl-grpo-7b={A}/grpo-7b '
    f'--max-model-len 2048 --gpu-memory-utilization 0.85',
    shell=True,
    stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT)

import os
os.makedirs('results-analysis/aug20', exist_ok=True)
!pip show vllm | grep Version | tee results-analysis/aug20/taskD-session-vllm-version.txt

for _ in range(150):  # model download + tp startup can take ~10-15 min
    try:
        urllib.request.urlopen('http://localhost:8000/health')
        print('vLLM ready')
        break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError('vLLM failed — check /kaggle/working/vllm.log')

In [ ]:
# Cell 3 — eval all three arms through the one session, then paired bootstrap
!python -m connections_rl.eval.run --config results-analysis/aug20/taskD_eval_ckpt50.yaml
!python results-analysis/aug20/taskD_paired_groups.py
!zip -qr /kaggle/working/taskD-session.zip \
    results-analysis/aug20/taskD-session \
    results-analysis/aug20/taskD-session-vllm-version.txt
print('done — download /kaggle/working/taskD-session.zip from the output panel')
# Sanity marks (do NOT rerun on drift — new session, drift on base/sft-like arms
# is expected; exact GRPO reproduction has been the historical pattern):
#   base ~ 0.160 groups (session A: 0.16049), grpo-final ~ 0.025 (A: 0.02469).
#   grpo-ckpt50 is the unknown this run exists to measure.

In [ ]:
# Cell 4 — durable copy: Hub results dataset + local download link
from huggingface_hub import HfApi
from IPython.display import FileLink, display
HF_USER = 'jacksonlukas'
api = HfApi()
repo = f'{HF_USER}/connections-rl-results'
api.upload_folder(folder_path='results-analysis/aug20/taskD-session',
                  repo_id=repo, repo_type='dataset',
                  path_in_repo='aug20/taskD-session')
api.upload_file(path_or_fileobj='results-analysis/aug20/taskD-session-vllm-version.txt',
                repo_id=repo, repo_type='dataset',
                path_in_repo='aug20/taskD-session-vllm-version.txt')
print(f'pushed -> huggingface.co/datasets/{repo}/tree/main/aug20/taskD-session')
display(FileLink('/kaggle/working/taskD-session.zip'))

**Afterwards:** unzip `taskD-session.zip` into
`~/Desktop/connections-rl/results-analysis/aug20/` on the Mac (so
`taskD-session/` sits beside the other aug20 files) and tell the agent — it
re-derives every number from the records and writes the Task D report.

**Task F (optional, only after reading D's numbers):** this same session type
can run `python -m connections_rl.train.grpo --config
results-analysis/aug20/taskF_grpo-7b-beta01.yaml` (~5.5 T4-h; needs the
2xT4 accelerate config per the training notebooks). One retry maximum if it
dies; A–E are already banked.